# Ten Trails — full MF6 model

Builds every package and writes the files. Nothing is run for you: the model does not
converge past period 2, and that is the thing to look at.

Companion: `usg_import_workbook.ipynb` explores the USG side and diagnoses convergence.


In [1]:
from pathlib import Path
import warnings, logging

import numpy as np
import pandas as pd
import flopy
import myflopy as mf
from myflopy.modflow.usg import read_usg

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)-7s %(message)s')

USG_DIR = Path('/home/lukem/models/mf6/Ten Trails/SSPA/Model_FINAL_ForLuke/14-Calib40_Iter01_DV')
NAM     = USG_DIR / 'flow-tt01_USE_5yr.nam'   # 72 periods; the one d_model.bat runs
GSF     = USG_DIR / 'flow-tt01.gsf'
CRS     = 'EPSG:2927'                         # WA State Plane South, ft
START   = '2017-10-01'                        # the DISU date column is unusable, see report()
NAME    = 'tentrails'
WORK    = Path('/tmp/tentrails_mf6')

## 1 · The model


In [2]:
usg = read_usg(NAM, gsf=GSF, crs=CRS)
usg

INFO    reading MODFLOW-USG model flow-tt01_USE_5yr.nam (BAS6, SMS, DISU, OC, RCH, WEL, DRN, CHD, LPF, CLN, GHB, ETS, HFB6)
INFO    building the Voronoi grid
INFO    Voronoi grid ready: 9405 cells
WARNING flow-tt01_5yr.dis carries a date on each stress-period line, but the dates do not increase (2023-10-31 then 2023-01-31) -- so they cannot give a start date. Pass start_date_time= to to_mf6() to set one.
INFO    read 5 layers x 9405 cells, 72 periods, 4 boundary package(s), CLN with 804 nodes


UsgModel('flow-tt01_USE_5yr.nam', 5 layers x 9405 cells, 72 periods, packages=['CHD', 'DRN', 'GHB', 'WEL'])

In [3]:
# Every package. `fix_for_mf6` makes the edits MF6 requires to accept the model at all
# (GHB heads raised to their cell bottom, CHD dropped in the periods where it sits below);
# each one is logged. Drop it to see what MF6 refuses, or call usg.validate() first.
sim = usg.to_mf6(NAME, start_date_time=START, fix_for_mf6=True)
sim

WARNING CHD: omitted 348 record-period(s) whose head sits below the cell bottom -- inert in MODFLOW-USG, rejected by MODFLOW 6 (fix_for_mf6=True)
WARNING GHB: raised 6 boundary head(s) to their cell bottom so MODFLOW 6 accepts them (fix_for_mf6=True)
INFO    ETS -> list-based EVT: 72 periods x 9090 columns = 654480 records (nseg=2)
INFO    converted flow-tt01_USE_5yr.nam to MODFLOW 6: 11 packages on a 5 x 9405 DISV grid, 72 periods


Name,Summary
tentrails,gwf; packages=11
Name,Summary
tdis,flopy.ModflowTdis; 4 options
ims,build_ims; 16 options


## 2 · Write it

~20 s. The segmented EVT is list-based and 61 MB — that is the physics, not a format choice.


In [4]:
import shutil
shutil.rmtree(WORK, ignore_errors=True)

project = mf.Project(WORK, name=NAME)
project.add_simulation(sim)
run = project.prepare_run('full', sim)

In [5]:
ims2 = mf.ims(
    name='ims',
    models=[NAME],
    complexity='COMPLEX',
    outer_maximum=500,
    inner_maximum=200,
    linear_acceleration='BICGSTAB',
    relaxation_factor=0.0,
    outer_dvclose=1e-3,
    inner_dvclose=1e-4,
    print_option='SUMMARY',
    under_relaxation='DBD',
    under_relaxation_theta=0.7,
    under_relaxation_kappa=0.1,
    under_relaxation_gamma=0.2,
    under_relaxation_momentum=0.0,
    backtracking_number=10,
    backtracking_tolerance=1.1,
    backtracking_reduction_factor=0.2,
    backtracking_residual_limit=100,
)
new_spec = run.spec.with_package(ims2)
tuned_run = project.prepare_run('tuned', new_spec, overwrite=True)
tdis = tuned_run.model().sim.tdis.perioddata.get_data()
tdis['nstp'] = 10
tuned_run.write()

writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims...
  writing model tentrails...
    writing model name file...
    writing package disv...
    writing package npf...
    writing package ic...
    writing package sto...
    writing package chd...
INFORMATION: maxbound in ('', 'chd', 'dimensions') changed to 66 based on size of stress_period_data
    writing package drn...
INFORMATION: maxbound in ('', 'drn', 'dimensions') changed to 205 based on size of stress_period_data
    writing package ghb...
INFORMATION: maxbound in ('', 'ghb', 'dimensions') changed to 88 based on size of stress_period_data
    writing package rch...
    writing package evt...
INFORMATION: maxbound in ('', 'evt', 'dimensions') changed to 9090 based on size of stress_period_data
    writing package hfb...
INFORMATION: maxhfb in ('', 'hfb', 'dimensions') changed to 28 based on size of stress_period_data
    writing package oc...


PosixPath('/tmp/tentrails_mf6/runs/tuned')

In [6]:
m = tuned_run.model()

In [33]:
m.plot.map(per=13, layer=-1, show_mounding=True, show_layer_elevs=True).show()

In [7]:
m.packages.drn.inputs.mosaic().show()

In [22]:
m.gwf.drn.stress_period_data.data[0]

rec.array([((2, 650), 550.        , 1.723579e+05, ''),
           ((2, 652), 550.        , 1.723579e+05, ''),
           ((2, 658), 550.        , 1.723579e+05, ''),
           ((2, 864), 550.        , 1.723579e+05, ''),
           ((2, 1101), 550.        , 1.723579e+05, ''),
           ((2, 1103), 550.        , 1.723579e+05, ''),
           ((2, 1106), 550.        , 1.723579e+05, ''),
           ((2, 1333), 550.        , 1.723579e+05, ''),
           ((2, 1334), 550.        , 1.723579e+05, ''),
           ((2, 1335), 550.        , 1.723579e+05, ''),
           ((2, 1338), 550.        , 1.723579e+05, ''),
           ((2, 1339), 550.        , 1.723579e+05, ''),
           ((2, 1341), 550.        , 1.723579e+05, ''),
           ((2, 1342), 550.        , 1.723579e+05, ''),
           ((2, 1343), 550.        , 1.723579e+05, ''),
           ((2, 1344), 550.        , 1.723579e+05, ''),
           ((2, 1351), 550.        , 1.723579e+05, ''),
           ((2, 1597), 550.        , 1.723579e+05, '

In [51]:
feat: mf.modflow.usg.cln.ClnFeature = usg.cln.streams[0]

feat.

ClnFeature(index=4, kind='stream', nodes=array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 87, 88, 89, 90,
       91, 92, 93, 94, 95, 96, 97]), gwf_nodes=array([9088, 9087, 9092, 9093, 9094, 9134, 9177, 9217, 9216, 9243, 9244,
       9242, 9241, 9240, 9239, 9227, 9188, 9146, 9096, 9042, 9043, 9041,
       9040, 9044, 9061, 9105, 9104, 9103, 9058, 8998, 8941, 8940, 8939,
       8927, 8926, 8857, 8773, 8674, 8563, 8431, 8277, 8106, 8105, 7915,
       7916, 7917, 7914, 9085, 9032, 8973, 8912, 8838, 8751, 8650, 8646,
       8531, 8532, 8392]), layers=array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), cells=array([9087, 9086, 9091, 9092, 9093, 9133, 9176, 9216, 9215, 9242, 9243,
       9241,

## 3 · Run it yourself

The cell above prints the command. A terminal is better than a notebook here —
you get the outer-iteration table as it scrolls.

Or from here, with live output:


In [ ]:
success, report = run.execute(silent=False)
print(success)

## 4 · Afterwards


In [ ]:
model = run.model(NAME)
model.file_summary()

In [ ]:
# where it stopped
hds = flopy.utils.HeadFile(ws / f'{NAME}.hds')
kk = hds.get_kstpkper()
print(f'{len(kk)} records, last = period {kk[-1][1] + 1} of {usg.nper}')